In [3]:
# === A: IMPORTS & SEED ===
import os, json, random
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchvision import transforms, datasets
from PIL import Image
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


Device: cuda


In [9]:
# === B: PATHS & CLASS NAMES ===
# 👉 chỉnh hai dòng này cho đúng của bạn
CKPT_PATH = Path("./runs/mtl_efficientnet_b0_01/checkpoints/mtl_effcientnet_b0_best.pt")  # ví dụ
RUNS_META = Path("./runs_meta/class_names.json")  # nếu có

# Nếu có file class_names.json (sinh ra lúc train) thì lấy cái đó làm chuẩn
if RUNS_META.exists():
    class_names = json.loads(RUNS_META.read_text(encoding="utf-8"))
    print("Loaded class_names from runs_meta:", len(class_names))
else:
    # fallback: đọc từ checkpoint (nếu người train đã lưu)
    ckpt = torch.load(CKPT_PATH, map_location="cpu")
    if "class_names" in ckpt:
        class_names = ckpt["class_names"]
        print("Loaded class_names from CKPT:", len(class_names))
    else:
        raise RuntimeError("Không tìm thấy class_names cả ở runs_meta lẫn checkpoint.")

NUM_CLASSES = len(class_names)
CLASS_TO_IDX = {c:i for i,c in enumerate(class_names)}


Loaded class_names from CKPT: 33


In [10]:
# === C: TRANSFORMS ===
IMG_SIZE = 224

# Train augment (nếu cần train lại) – còn đánh giá thì dùng TEST_TFM
TRAIN_TFM = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

TEST_TFM = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

print("Transforms OK.")


Transforms OK.


In [11]:
# === D: BUILD MODEL & LOAD CKPT (STRICT) ===
def build_model(num_classes: int, pretrained: bool = False):
    """
    pretrained=True: dùng khi bạn FINETUNE từ ImageNet.
    Khi LOAD checkpoint đã train rồi -> luôn khởi tạo weights=None.
    """
    if pretrained:
        m = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    else:
        m = efficientnet_b0(weights=None)
    in_f = m.classifier[1].in_features
    m.classifier[1] = nn.Linear(in_f, num_classes)
    return m

# Khởi tạo model cho LOAD (weights=None) để head khớp tuyệt đối.
model = build_model(NUM_CLASSES, pretrained=False).to(device)
ckpt = torch.load(CKPT_PATH, map_location=device)
state = ckpt["net"] if "net" in ckpt else ckpt

missing, unexpected = model.load_state_dict(state, strict=True)
print("Loaded CKPT strict. missing:", missing, " unexpected:", unexpected)
model.eval()


RuntimeError: Error(s) in loading state_dict for EfficientNet:
	Missing key(s) in state_dict: "features.0.0.weight", "features.0.1.weight", "features.0.1.bias", "features.0.1.running_mean", "features.0.1.running_var", "features.1.0.block.0.0.weight", "features.1.0.block.0.1.weight", "features.1.0.block.0.1.bias", "features.1.0.block.0.1.running_mean", "features.1.0.block.0.1.running_var", "features.1.0.block.1.fc1.weight", "features.1.0.block.1.fc1.bias", "features.1.0.block.1.fc2.weight", "features.1.0.block.1.fc2.bias", "features.1.0.block.2.0.weight", "features.1.0.block.2.1.weight", "features.1.0.block.2.1.bias", "features.1.0.block.2.1.running_mean", "features.1.0.block.2.1.running_var", "features.2.0.block.0.0.weight", "features.2.0.block.0.1.weight", "features.2.0.block.0.1.bias", "features.2.0.block.0.1.running_mean", "features.2.0.block.0.1.running_var", "features.2.0.block.1.0.weight", "features.2.0.block.1.1.weight", "features.2.0.block.1.1.bias", "features.2.0.block.1.1.run